**Structured Streaming Basics**

**Topic**: Streaming | Exercises: 8 | Total Time: ~90 min

Practice the Structured Streaming foundation on Databricks: readStream and writeStream against Delta tables, choosing triggers (processingTime vs availableNow), picking the right output mode (append / complete / update), naming queries and reading lastProgress, and deduplicating streams. Every query uses trigger(availableNow=True) and awaitTermination() so the notebook terminates cleanly on Free Edition.

**Solutions**: If stuck, see solutions/structured-streaming-basics-solutions.py for hints and answers.

**Key concepts**:

spark.readStream.table(...) reads a Delta table as a stream of newly committed rows
trigger(availableNow=True) processes everything available and stops (use this on Free Edition)
Output modes: append (only new rows), complete (full result table), update (changed rows)
Streaming queries always need a checkpointLocation for state and offset tracking
.queryName(...) makes a stream identifiable; query.lastProgress exposes runtime metrics
dropDuplicates(["key"]) deduplicates a stream using watermarked state

In [0]:
CATALOG = "db_code"
SCHEMA = "structured_streaming_basics"
CHECKPOINT_BASE = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

**Exercise 1: Read a Delta Table as a Stream, Write to Another Delta Table**
**Difficulty**: Easy | **Time**: ~10 min

The foundational Structured Streaming pattern: treat an existing Delta table as a stream source and write the streamed rows to a target Delta table in append mode.

**Source **(db_code.structured_streaming_basics.ex1_source): 35 events with columns event_id STRING, event_type STRING, user_id STRING, amount DOUBLE, event_ts TIMESTAMP.

**Target**: Write to db_code.structured_streaming_basics.ex1_target. Expected 35 rows.

**Requirements**:

- Read ex1_source with spark.readStream (Delta source)
- Write to ex1_target as Delta with output mode append
- Set a checkpointLocation under CHECKPOINT_BASE (e.g., f"{CHECKPOINT_BASE}/ex1")
- Use .trigger(availableNow=True) so the stream terminates after one batch
- Call .awaitTermination() so the next cell runs only after the stream finishes

**Constraints**:

Use trigger(availableNow=True) (NOT trigger(once=True), NOT trigger(processingTime=...))
Checkpoint path must be in a Volume

In [0]:
df=spark.readStream.format("delta").table(f"{CATALOG}.{SCHEMA}.ex1_source")

df.writeStream.format("delta")\
    .outputMode("append")\
    .option('checkpointLocation',f"{CHECKPOINT_BASE}/ex_1")\
    .trigger(availableNow=True)\
    .toTable(f"{CATALOG}.{SCHEMA}.ex1_target")\
    .awaitTermination()    



In [0]:
# Validate Exercise 1
result = spark.table(f"{CATALOG}.{SCHEMA}.ex1_target")

assert result.count() == 35, f"Expected 35 rows, got {result.count()}"
assert "event_id" in result.columns, "Missing event_id column"
assert "amount" in result.columns, "Missing amount column"
assert result.filter("event_id = 'EV-001'").count() == 1, "EV-001 should exist"
assert result.filter("event_id = 'EV-035'").count() == 1, "EV-035 should exist"
ev3_amount = result.filter("event_id = 'EV-003'").select("amount").collect()[0][0]
assert ev3_amount == 120.50, f"EV-003 amount should be 120.50, got {ev3_amount}"

print("Exercise 1 passed!")